## Decode and encode trajectory for crop

In [1]:
%load_ext autoreload
%autoreload 2
import os
import functools
from pathlib import Path
import numpy as np
import re
import matplotlib.pyplot as plt

from data_loader_jsonl import JSONLDataset, ValidDataset
from data_augmentations import CropStart, CropMiddle

imginfo = lambda img: print(type(img), img.dtype, img.shape, img.min(), img.max())

return_depth = False
dataset_location = "./data/clevr-real-block-v3"
dataset_location = Path(dataset_location)

model_location = Path("./data/models/")
model_path = model_location / "clevr-act-7-depth_text_aug" / "checkpoint-4687"

save_folder = "./data/prediction_middle/"
Path(save_folder).mkdir(parents=True, exist_ok=True)

if "depth_depth_l40" in str(model_path):
    return_depth = True

print("dataset_location", dataset_location)
if model_path.is_dir():
    print("moadel_path is", model_path)

dataset_location data/clevr-real-block-v3
moadel_path is data/models/clevr-act-7-depth_text_aug/checkpoint-4687


In [4]:
# Load precomputed results.
# Ignore all tokens from the language model except loc tokens.

episode_i = 0
crop_size = 400

with open(save_folder + f"tokens_{episode_i:04d}_{crop_size:04d}.npy", "rb") as f:
    decoded_str = np.load(f)
    scores = np.load(f)
print(decoded_str)
imginfo(scores)

token_pos = 0 # 0 - start y, 1 - start x, 6 - end y, 7 - end x
probas = scores[token_pos, 0]

# === softmax
temp = 10
probas = np.exp(probas / temp)
probas = probas / (np.sum(probas) + 1e-3)

fig, ax = plt.subplots(figsize=(8, 3))
#ax.scatter(tokens, probas, color='skyblue', alpha=0.3)
ax.plot(np.arange(0, 1024), probas)
ax.set_xlim(0, 1024)
ax.set_ylabel("Score")
ax.set_title("Loc tokens")
ax.set_xlabel("x")

In [5]:
# use a standard function to get image coors from string

from utils_vis import get_standard_camera
from utils_traj_tokens import decode_caption_xyzrotvec2

crop_size = 500
test_dataset = JSONLDataset(
    jsonl_file_path=f"{dataset_location}/_annotations.valid.jsonl",
    image_directory_path=f"{dataset_location}/dataset",
    clean_prompt=True,
    #augment_crop=CropStart(crop_size=1000, valid=False),
    augment_crop=CropMiddle(crop_size=crop_size, object_size=20, valid=True),
    return_depth=return_depth
)

batch_entry = test_dataset[episode_i]
if return_depth:
    (depth, image), sample = batch_entry
else:
    image, sample = batch_entry

image_width, image_height = image.size  # PIL image
camera = get_standard_camera(image_width, image_height)

# curve_25d has shape (2, 3) = 
# x1, y1, d1, 
# x2, y2, d2
# quat_c has shape (2, 4) =
# qw, qx, qy, qz
# qw, qx, qy, qz
curve_25d, quat_c = decode_caption_xyzrotvec2(str(decoded_str), camera=camera)

NameError: name 'decoded_str' is not defined

In [ ]:
print(curve_25d)
print(quat_c)

In [ ]:
# convert string tokens to int
# depth in cm, x and y in [0, 1023]
loc_strings = [int(x) for x in re.findall(r"<(?:loc|seg)(-?\d+)>", str(decoded_str))]
loc_strings = np.array(loc_strings)
loc_strings = loc_strings.reshape(-1, 6) # each row contains (h w d r0 r1 r2)

# convert from maniskill (1024, 1024) to (image_height, image_width)
loc_h = (loc_strings[:, 0]/(1024-1)*image_height).round().astype(int) # (obj_start_y obj_end_y)
loc_w = (loc_strings[:, 1]/(1024-1)*image_width).round().astype(int) # (obj_start_x obj_end_x)
print(loc_h)

In [ ]:
keypoint = 0
coor = 1
assert np.allclose(loc_h[keypoint], curve_25d[keypoint, coor], atol=1.0)

keypoint = 1
coor = 1
assert np.allclose(loc_h[keypoint], curve_25d[keypoint, coor], atol=1.0)

In [ ]:
# now encode it back, like I do in crop_augment
# TODO split encode function into 3d -> 2d and 2d -> string